# 09 -- Evaluación robusta del baseline de Nivel 1 (fase A)

Noveno notebook de la cadena. **No entrena nada**: toma la evaluación walk-forward de
`08_nivel1_demanda` (`evaluacion_nivel_1.parquet`) y le pregunta lo que 08 no responde:
*¿la diferencia contra la media móvil es real o es ruido?*

Por qué hace falta: las rondas 1-5 de búsqueda (383 corridas) se evaluaron contra los mismos
4 folds que se reportan, y las diferencias son de 0.5-2.5%. Aquí se mide con IC 95% por
bootstrap de **productos** (las filas de un mismo producto están correlacionadas), por fold,
dejando cada fold afuera, y mirando cuánto del error se concentra en pocos productos.

Todo lo que se decide sale de `decision_09.json` (reglas mecánicas en `evaluar_nivel1.veredicto`),
que leerán 10-13. Cada corrida se registra en MLflow (experimento `EDA-fix-nivel1-09-evaluacion-robusta`).

Cadena: ... -> `07_matriz_as_of` -> `08_nivel1_demanda` -> **`09_evaluacion_robusta`** -> 10-13

In [1]:
import sys
sys.path.insert(0, '.')
from common_priorizacion import DW, registrar_huella, verificar_huella
import evaluar_nivel1 as ev

import json
import pandas as pd

pd.set_option('display.width', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

B, SEMILLA = 2000, 0                     # remuestreos bootstrap y semilla
COL_Y, COL_MODELO, COL_PISO = 'target_demanda_15d', 'pred_q50', 'pred_media_movil_15d'
EXPERIMENTO = 'EDA-fix-nivel1-09-evaluacion-robusta'

verificar_huella('huella_08.json', ['evaluacion_nivel_1.parquet'])
d = pd.read_parquet(f'{DW}/evaluacion_nivel_1.parquet')
omitidas = d['rama'].isna()
print(f'filas sin predicción (rama x fold omitida por < {ev.MIN_FILAS_ENTRENAMIENTO} filas de entrenamiento tras el purge): {int(omitidas.sum()):,}')
d = d.loc[~omitidas].copy()
RAMAS = ['intermitente', 'suave_no_perecedero', 'suave_perecedero']
assert set(d['rama']) == set(RAMAS), set(d['rama'])
print(d.groupby('rama').agg(filas=('codigo_item', 'size'), productos=('codigo_item', 'nunique')))

Huella verificada OK contra huella_08.json.
filas sin predicción (rama x fold omitida por < 200 filas de entrenamiento tras el purge): 546


                      filas  productos
rama                                  
intermitente         232704       3050
suave_no_perecedero    2410         60
suave_perecedero       1044         22


## 1. Mejora sobre el piso con IC 95% (bootstrap por producto)

`mejora_rel = (WAPE_piso - WAPE_modelo) / WAPE_piso`; positivo = el modelo gana. Cortes: todos los
folds, cada fold solo, y dejando cada fold afuera.

In [2]:
tablas = {r: ev.evaluar_rama(d[d['rama'] == r], COL_Y, COL_MODELO, COL_PISO, B, SEMILLA) for r in RAMAS}
for r, t in tablas.items():
    print(f'\n=== {r} ===')
    print(t[['corte', 'n_productos', 'wape_modelo', 'wape_piso', 'mejora_rel', 'ic_bajo', 'ic_alto', 'p_mejora_positiva']].to_string(index=False))


=== intermitente ===
      corte  n_productos  wape_modelo  wape_piso  mejora_rel  ic_bajo  ic_alto  p_mejora_positiva
      todos         3050        0.433      0.438       0.012   -0.061    0.056              0.729
solo_fold_2         2594        0.449      0.408      -0.100   -0.332    0.029              0.372
solo_fold_3         2803        0.466      0.490       0.049    0.009    0.077              0.990
solo_fold_4         2888        0.422      0.451       0.064    0.051    0.077              1.000
solo_fold_5         3050        0.410      0.435       0.057    0.044    0.070              1.000
 sin_fold_2         3050        0.426      0.452       0.057    0.045    0.068              1.000
 sin_fold_3         3050        0.427      0.430       0.005   -0.077    0.053              0.696
 sin_fold_4         3050        0.436      0.434      -0.006   -0.102    0.052              0.409
 sin_fold_5         2888        0.443      0.439      -0.008   -0.106    0.051              0.38

## 2. Veredicto mecánico

`supera_con_ic`: el IC95 de la mejora total queda por encima de 0. `robusto`: además lo sigue
estando sin el fold más favorable. Ninguna de las dos se escribe a mano.

In [3]:
veredictos = {r: ev.veredicto(t) for r, t in tablas.items()}
print(pd.DataFrame(veredictos).T[['mejora_rel', 'ic_bajo', 'ic_alto', 'supera_con_ic',
                                    'fold_mas_favorable', 'mejora_sin_fold_favorable', 'robusto']])
for r, v in veredictos.items():
    if v['supera_con_ic']:
        print(f'{r}: SUPERA el piso con IC95 > 0' + (' y es ROBUSTO.' if v['robusto'] else ', pero NO es robusto sin el fold más favorable.'))
    elif v['mejora_rel'] > 0:
        print(f'{r}: mejora puntual {v["mejora_rel"]:.1%} pero el IC95 [{v["ic_bajo"]:.1%}, {v["ic_alto"]:.1%}] incluye 0 -> no distinguible del piso.')
    else:
        print(f'{r}: NO supera el piso (mejora puntual {v["mejora_rel"]:.1%}, IC95 [{v["ic_bajo"]:.1%}, {v["ic_alto"]:.1%}]).')

                    mejora_rel ic_bajo ic_alto supera_con_ic fold_mas_favorable mejora_sin_fold_favorable robusto
intermitente             0.012  -0.061   0.056         False                  4                    -0.006   False
suave_no_perecedero      0.010  -0.038   0.049         False                  5                    -0.036   False
suave_perecedero        -0.025  -0.236   0.020         False                  5                    -0.079   False
intermitente: mejora puntual 1.2% pero el IC95 [-6.1%, 5.6%] incluye 0 -> no distinguible del piso.
suave_no_perecedero: mejora puntual 1.0% pero el IC95 [-3.8%, 4.9%] incluye 0 -> no distinguible del piso.
suave_perecedero: NO supera el piso (mejora puntual -2.5%, IC95 [-23.6%, 2.0%]).


## 3. ¿El resultado lo mueven pocos productos?

WAPE pesa por volumen, así que un producto de altísima rotación con un error grande puede
decidir el agregado. Se mide, por fold, qué parte del exceso de error positivo (modelo peor que
el piso) concentran los 10 peores productos, y cuánto mejora la comparación al quitar el peor.

In [4]:
conc = {r: ev.concentracion_exceso(d[d['rama'] == r], COL_Y, COL_MODELO, COL_PISO, k=10) for r in RAMAS}
for r, c in conc.items():
    print(f'\n=== {r} ===')
    print(c.to_string(index=False))

sin_peor = {r: ev.mejora_sin_peor_producto(d[d['rama'] == r], COL_Y, COL_MODELO, COL_PISO, B, SEMILLA) for r in RAMAS}
print('\nSensibilidad: quitando el producto que más exceso de error concentra (todos los folds)')
print(pd.DataFrame(sin_peor).T[['producto_excluido', 'wape_modelo', 'wape_piso', 'mejora_rel', 'ic_bajo', 'ic_alto']])


=== intermitente ===
 fold_id  exceso_neto  exceso_positivo peor_producto  exceso_peor_producto  share_top_k  peor_explica_mas_que_neto
       2   21,471.500       33,779.158         P1632            25,099.699        0.832                       True
       3   -5,621.799        5,160.087         P3937             1,608.577        0.523                      False
       4  -11,907.536        3,881.519          P641               190.645        0.275                      False
       5  -12,613.113        4,094.083         P1632               532.497        0.347                      False

=== suave_no_perecedero ===
 fold_id  exceso_neto  exceso_positivo peor_producto  exceso_peor_producto  share_top_k  peor_explica_mas_que_neto
       2      791.116        1,733.338         P1451               371.736        0.792                      False
       3      570.528          784.097         P1451               256.164        0.850                      False
       4      -29.647        


Sensibilidad: quitando el producto que más exceso de error concentra (todos los folds)
                    producto_excluido wape_modelo wape_piso mejora_rel ic_bajo ic_alto
intermitente                    P1632       0.434     0.456      0.049   0.040   0.057
suave_no_perecedero              P488       0.191     0.194      0.017  -0.027   0.054
suave_perecedero                P1805       0.139     0.137     -0.010  -0.197   0.023


In [5]:
# Diagnóstico del peor caso de la rama intermitente: ¿qué series lo producen?
peor = sin_peor['intermitente']['producto_excluido']
det = (d[(d['rama'] == 'intermitente') & (d['codigo_item'] == peor)]
       .assign(e_modelo=lambda x: (x[COL_Y] - x[COL_MODELO]).abs(), e_piso=lambda x: (x[COL_Y] - x[COL_PISO]).abs())
       .groupby(['fold_id', 'sucursal'], observed=True)
       .agg(filas=('e_modelo', 'size'), y_medio=(COL_Y, 'mean'), pred_modelo=(COL_MODELO, 'mean'),
            pred_piso=(COL_PISO, 'mean'), e_modelo=('e_modelo', 'sum'), e_piso=('e_piso', 'sum')))
det['exceso'] = det['e_modelo'] - det['e_piso']
print(f'producto {peor}: series con mayor exceso de error')
print(det.sort_values('exceso', ascending=False).head(6))
tr_max = d.loc[(d['rama'] == 'intermitente'), 'trail_15'].quantile(0.999)
print(f'\np99.9 de trail_15 en la rama: {tr_max:,.0f} -- las series de ese producto están muy por encima, en una zona con muy pocas filas de entrenamiento')

producto P1632: series con mayor exceso de error
                   filas   y_medio  pred_modelo  pred_piso   e_modelo    e_piso     exceso
fold_id sucursal                                                                          
2       PRINCIPAL     13 2,807.692      418.773  2,787.096 31,055.949 5,956.250 25,099.699
3       PRINCIPAL      5 3,131.200    2,413.894  2,894.950  3,586.528 2,759.250    827.278
5       PRINCIPAL     12 3,004.333    2,565.569  2,917.938  7,470.247 6,937.750    532.497
4       PRINCIPAL     10 2,625.600    2,321.301  2,810.475  5,841.639 6,038.250   -196.611

p99.9 de trail_15 en la rama: 340 -- las series de ese producto están muy por encima, en una zona con muy pocas filas de entrenamiento


## 4. Política de holdout y decisión

La búsqueda de las fases siguientes solo puede usar los folds de `FOLDS_BUSQUEDA` y reportar el
`FOLD_HOLDOUT` una vez por fase. **Limitación**: el fold 5 ya fue visto por las rondas 1-5, así que
no es un holdout virgen; un holdout limpio requiere ventas posteriores a 2025-12-31.

In [6]:
decision = {
    'notebook': '09_evaluacion_robusta',
    'fecha_generacion': pd.Timestamp.now().isoformat(),
    'metodo': {'bootstrap_B': B, 'semilla': SEMILLA, 'unidad_remuestreo': 'codigo_item',
               'comparacion': f'{COL_MODELO} vs {COL_PISO}'},
    'politica_holdout': ev.POLITICA_HOLDOUT,
    'ramas': {r: {**veredictos[r],
                  'producto_que_mas_pierde': sin_peor[r]['producto_excluido'],
                  'mejora_sin_ese_producto': sin_peor[r]['mejora_rel'],
                  'ic_sin_ese_producto': [sin_peor[r]['ic_bajo'], sin_peor[r]['ic_alto']]}
              for r in RAMAS},
}
with open(f'{DW}/decision_09.json', 'w') as f:
    json.dump(decision, f, indent=2, ensure_ascii=False)

todo = pd.concat([t.assign(rama=r) for r, t in tablas.items()], ignore_index=True)
todo.to_csv(f'{DW}/resultados_evaluacion_robusta.csv', index=False)
pd.concat([c.assign(rama=r) for r, c in conc.items()]).to_csv(f'{DW}/concentracion_error_por_fold.csv', index=False)

huella = {'notebook': '09_evaluacion_robusta', 'fecha_generacion': pd.Timestamp.now().isoformat(),
          'parametros': {'B': B, 'semilla': SEMILLA},
          'entradas': registrar_huella(['evaluacion_nivel_1.parquet']),
          'salidas': registrar_huella(['decision_09.json', 'resultados_evaluacion_robusta.csv', 'concentracion_error_por_fold.csv'])}
with open(f'{DW}/huella_09.json', 'w') as f:
    json.dump(huella, f, indent=2, ensure_ascii=False)
print('guardado: decision_09.json, resultados_evaluacion_robusta.csv, concentracion_error_por_fold.csv, huella_09.json')

guardado: decision_09.json, resultados_evaluacion_robusta.csv, concentracion_error_por_fold.csv, huella_09.json


## 5. Registro en MLflow

Un run por rama: parámetros del método, la mejora con su IC, el veredicto y el peor producto.
Falla en voz alta si el servidor no responde.

In [7]:
for r in RAMAS:
    v, s = veredictos[r], sin_peor[r]
    run_id = ev.registrar_mlflow(
        EXPERIMENTO, f'evaluacion_robusta_{r}',
        tags={'fase': 'A', 'rama': r, 'rama_git': 'fix/eda-modelado-nivel1',
              'matriz_huella': huella['entradas']['evaluacion_nivel_1.parquet']},
        params={'bootstrap_B': B, 'semilla': SEMILLA, 'comparacion': f'{COL_MODELO} vs {COL_PISO}',
                'producto_que_mas_pierde': s['producto_excluido'], 'fold_mas_favorable': v['fold_mas_favorable']},
        metrics={'wape_modelo': tablas[r].iloc[0]['wape_modelo'], 'wape_piso': tablas[r].iloc[0]['wape_piso'],
                 'mejora_rel': v['mejora_rel'], 'ic_bajo': v['ic_bajo'], 'ic_alto': v['ic_alto'],
                 'supera_con_ic': int(v['supera_con_ic']), 'robusto': int(v['robusto']),
                 'mejora_sin_fold_favorable': v['mejora_sin_fold_favorable'],
                 'mejora_sin_peor_producto': s['mejora_rel'], 'ic_bajo_sin_peor_producto': s['ic_bajo']},
        artefactos=[f'{DW}/decision_09.json', f'{DW}/resultados_evaluacion_robusta.csv'])
    print(f'{r}: run {run_id}')

/home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026/09/24 18:35:47 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /home/ddelgadillo@redcorporativa.corpoica.org.co/miniforge3/envs/etl/lib/python3.11/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


🏃 View run evaluacion_robusta_intermitente at: http://127.0.0.1:5000/#/experiments/14/runs/985a035877df41ff99bde1b246fb5c47
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/14
intermitente: run 985a035877df41ff99bde1b246fb5c47


🏃 View run evaluacion_robusta_suave_no_perecedero at: http://127.0.0.1:5000/#/experiments/14/runs/ed36db4b25fd495594abc6b5a57cc763
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/14
suave_no_perecedero: run ed36db4b25fd495594abc6b5a57cc763


🏃 View run evaluacion_robusta_suave_perecedero at: http://127.0.0.1:5000/#/experiments/14/runs/fdb43ff776de482f8eae41c951a08c95
🧪 View experiment at: http://127.0.0.1:5000/#/experiments/14
suave_perecedero: run fdb43ff776de482f8eae41c951a08c95
